# SIGMOD Exp 2: Patch IVMH at Zero History/Delta

This notebook:

1. Locates the two source Exp2 CSVs
2. Copies them into patched outputs
3. Re-runs only the `IVMH` point at `history_ratio = 0` / `delta_ratio = 0`
4. Overwrites the copied CSV rows for those two points
5. Re-renders the same Exp2 crossover plots from the patched CSVs

The zero-history and zero-delta traces are identical under this preset, so we run the point once and reuse it for both files.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib'])
print('done')


In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    apply_paper_style,
    current_run_stamp,
    ensure_dirs,
    run_checked,
)
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_crossover').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
SERVER_RESULT_DIR = ROOT / 'benches' / 'sigmod_server_result' / 'sigmod_exp2_crossover' / 'data'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TX_MAP = {'MarkTs': 'BuildSnap', 'DelSc': 'DeltaScan'}

CONFIG = {
    'warehouse_count': 7,
    'txn_count': 100,
    'bucket_num': 4096,
    'update_ratio': 0.0001,
    'probe_ratio': 0.001,
    'txn_gc_ratio': 0.05,
    'readable_every': 2,
    'repeat': 5,
    'trim': 1,
    'timeout_sec': 900,
}

SOURCE_HISTORY_NAME = 'sigmod_exp2_history_wc7_tc100_bn4096_ur0p0001_pr0p001_gc0p05_re2_rep5_20260409_0300.csv'
SOURCE_DELTA_NAME = 'sigmod_exp2_delta_wc7_tc100_bn4096_ur0p0001_pr0p001_gc0p05_re2_rep5_20260409_0300.csv'

PLOT_SERIES = [
    ('naive', ''),
    ('ivmh', ''),
    ('heap', 'Write Repair'),
    ('chain', 'Write Repair'),
    ('par', 'Write Repair'),
]
STYLE = {
    ('naive', ''): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', ''): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'Write Repair'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'Write Repair'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'Write Repair'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}

RUN_STAMP = current_run_stamp()
BASE_ARGS = [
    '--txn-count', str(CONFIG['txn_count']),
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--update-ratio', str(CONFIG['update_ratio']),
    '--probe-ratio', str(CONFIG['probe_ratio']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--readable-every', str(CONFIG['readable_every']),
]

print('ROOT :', ROOT)
print('BIN  :', BIN)
print('STAMP:', RUN_STAMP)


In [ ]:
print('Building htap_wkld...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_wkld'], ROOT)
print('Build OK')


In [ ]:
def find_source_csv(filename):
    candidates = [
        DATA_DIR / filename,
        SERVER_RESULT_DIR / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = sorted(ROOT.glob(f'benches/**/{filename}'))
    if matches:
        return matches[0]
    raise FileNotFoundError(filename)


def merge_args(base, extra):
    merged = {}
    for args in (base, extra):
        it = iter(args)
        for token in it:
            merged[token] = next(it)
    out = []
    for k, v in merged.items():
        out.extend([k, v])
    return out


def trim_trial_runs(df):
    trim = CONFIG['trim']
    if trim <= 0 or df.empty or 'trial' not in df.columns:
        return df
    totals = (
        df.groupby('trial', as_index=False)['duration_ms']
        .sum()
        .sort_values('duration_ms')
    )
    if len(totals) <= 2 * trim:
        return df
    keep = set(totals.iloc[trim:len(totals) - trim]['trial'])
    return df[df['trial'].isin(keep)].copy()


def collapse_repairs(df, table_type):
    collapsed = df.groupby(['table_type'], as_index=False)['duration_ms'].mean()
    collapsed['repair_type'] = ''
    return collapsed[['table_type', 'repair_type', 'duration_ms']]


def history_args(history_ratio):
    update_share = 0.20
    recent_read_share = 1.0 - update_share
    return [
        '--txn-update-ratio', str(update_share),
        '--txn-probe-ratio', str(recent_read_share / 2.0),
        '--txn-scan-ratio', str(recent_read_share / 2.0),
        '--txn-delta-ratio', '0.0',
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--scan-reuse-ratio', str(history_ratio),
        '--probe-history-ratio', str(history_ratio),
    ]


def run_ivmh_zero_point():
    args = merge_args(BASE_ARGS, history_args(0.0))
    trials = []
    for trial in range(CONFIG['repeat']):
        print(f'IVMH zero-point trial {trial + 1}/{CONFIG["repeat"]}')
        result = run_checked(
            [str(BIN), *args, '--table-type', 'ivmh'],
            ROOT,
            quiet=True,
            timeout=CONFIG['timeout_sec'],
        )
        df = parse_result(result.stdout, 'ivmh')
        if df.empty:
            raise RuntimeError('No parsed rows for IVMH zero-point run')
        df['tx_type'] = df['tx_type'].replace(TX_MAP)
        total = df.groupby('repair_type', as_index=False)['duration_ms'].sum()
        total['table_type'] = 'ivmh'
        total['trial'] = trial
        trials.append(total)
    df_all = trim_trial_runs(pd.concat(trials, ignore_index=True))
    df_avg = df_all.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].mean()
    out = collapse_repairs(df_avg, 'ivmh')
    out['total_ms'] = out['duration_ms'] / CONFIG['txn_count']
    return out.iloc[0].to_dict()


def patch_zero_row(df, x_col, zero_result):
    out = df.copy()
    out['repair_type'] = out['repair_type'].fillna('')
    out = out[~((out['table_type'] == 'ivmh') & (out[x_col].astype(float) == 0.0))].copy()
    replacement = {
        'table_type': 'ivmh',
        'repair_type': '',
        'duration_ms': float(zero_result['duration_ms']),
        'total_ms': float(zero_result['total_ms']),
        x_col: 0.0,
    }
    out = pd.concat([out, pd.DataFrame([replacement])], ignore_index=True)
    out = out.sort_values([x_col, 'table_type', 'repair_type']).reset_index(drop=True)
    return out


def plot_one(ax, df, x_col, xlabel, include_snap, legend_loc):
    for key in PLOT_SERIES:
        if not include_snap and key == ('naive', ''):
            continue
        label, color, linestyle, marker = STYLE[key]
        table_type, repair_type = key
        sub = df[(df['table_type'] == table_type) & (df['repair_type'] == repair_type)].sort_values(x_col)
        if sub.empty:
            continue
        ax.plot(
            sub[x_col],
            sub['total_ms'],
            color=color,
            linestyle=linestyle,
            marker=marker,
            linewidth=1.8,
            markersize=5,
            label=label,
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Duration (ms / tx)')
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    ax.legend(loc=legend_loc, ncol=1, framealpha=0.95)


def render_pair(df, x_col, xlabel, stem):
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 4.1), sharey=False)
    plot_one(axes[0], df, x_col, xlabel, True, 'center left')
    plot_one(axes[1], df, x_col, xlabel, False, 'upper left')
    fig.tight_layout()
    out_pdf = FIGS_DIR / f'{stem}_{RUN_STAMP}.pdf'
    fig.savefig(out_pdf, format='pdf')
    plt.show()
    print('Saved', out_pdf)


def render_single(df, x_col, xlabel, stem, include_snap):
    fig, ax = plt.subplots(1, 1, figsize=(5.0, 4.1))
    legend_loc = 'center left' if include_snap else 'upper left'
    plot_one(ax, df, x_col, xlabel, include_snap, legend_loc)
    fig.tight_layout()
    out_pdf = FIGS_DIR / f'{stem}_{RUN_STAMP}.pdf'
    fig.savefig(out_pdf, format='pdf')
    plt.show()
    print('Saved', out_pdf)


In [ ]:
source_history_path = find_source_csv(SOURCE_HISTORY_NAME)
source_delta_path = find_source_csv(SOURCE_DELTA_NAME)

history_df = pd.read_csv(source_history_path, keep_default_na=False)
delta_df = pd.read_csv(source_delta_path, keep_default_na=False)

zero_ivmh = run_ivmh_zero_point()
print('Patched zero-point result:')
display(pd.DataFrame([zero_ivmh]))

print('Original history IVMH@0:')
display(history_df[(history_df['table_type'] == 'ivmh') & (history_df['history_ratio'].astype(float) == 0.0)])
print('Original delta IVMH@0:')
display(delta_df[(delta_df['table_type'] == 'ivmh') & (delta_df['delta_ratio'].astype(float) == 0.0)])

patched_history = patch_zero_row(history_df, 'history_ratio', zero_ivmh)
patched_delta = patch_zero_row(delta_df, 'delta_ratio', zero_ivmh)
patched_history['history_pct'] = patched_history['history_ratio'].astype(float) * 100.0
patched_delta['delta_pct'] = patched_delta['delta_ratio'].astype(float) * 100.0

history_out = DATA_DIR / f'{source_history_path.stem}_ivmhzero_{RUN_STAMP}.csv'
history_latest = DATA_DIR / f'{source_history_path.stem}_ivmhzero_latest.csv'
delta_out = DATA_DIR / f'{source_delta_path.stem}_ivmhzero_{RUN_STAMP}.csv'
delta_latest = DATA_DIR / f'{source_delta_path.stem}_ivmhzero_latest.csv'

patched_history.drop(columns=['history_pct']).to_csv(history_out, index=False)
patched_history.drop(columns=['history_pct']).to_csv(history_latest, index=False)
patched_delta.drop(columns=['delta_pct']).to_csv(delta_out, index=False)
patched_delta.drop(columns=['delta_pct']).to_csv(delta_latest, index=False)

print('Saved', history_out)
print('Saved', history_latest)
print('Saved', delta_out)
print('Saved', delta_latest)


In [ ]:
render_pair(patched_history, 'history_pct', 'Historical Read Percentage (%)', 'exp2-history-pair-ivmhzero')
render_pair(patched_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'exp2-delta-pair-ivmhzero')


In [ ]:
render_single(patched_history, 'history_pct', 'Historical Read Percentage (%)', 'exp2-history-with-snap-ivmhzero', True)
render_single(patched_history, 'history_pct', 'Historical Read Percentage (%)', 'exp2-history-without-snap-ivmhzero', False)
render_single(patched_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'exp2-delta-with-snap-ivmhzero', True)
render_single(patched_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'exp2-delta-without-snap-ivmhzero', False)
